In [7]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("Booting Selenium with the corrected navigation flow...")

options = webdriver.ChromeOptions()
# options.add_argument('--headless=new')
options.add_argument('--window-size=1920,1080')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 10)

master_records = []

try:
    driver.get("https://jiji.ng/houses-apartments-for-rent")
    time.sleep(4)

    # 1. Open the Location Filter
    print("Opening location modal...")
    location_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//div[@filter-name='region'] | //div[contains(@class, 'filter') and contains(., 'Location')] | //div[contains(@class, 'b-filter-title')]")))
    driver.execute_script("arguments[0].click();", location_btn)
    time.sleep(2)

    # 2. Get inside Lagos to map the main districts first
    lagos_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(@class, 'b-region-item__name') and contains(text(), 'Lagos')]")))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", lagos_btn)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", lagos_btn)
    time.sleep(2)

    # Map the districts
    print("Mapping main districts...")
    district_elements = driver.find_elements(By.CLASS_NAME, "b-region-item__name")
    main_districts = [el.text.strip() for el in district_elements if el.text.strip() and not el.text.strip().lower().startswith('all ')]
    main_districts = sorted(list(set(main_districts)))
    
    print(f"Found {len(main_districts)} main districts. Starting extraction loop...")

    # 3. Hit back to reset to the Root menu (Nigeria View) so our loop has a consistent starting point
    back_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'b-popup-header__back')] | //*[name()='svg' and contains(@class, 'back')] | //div[contains(text(), 'Back')]")))
    driver.execute_script("arguments[0].click();", back_btn)
    time.sleep(2)

    # 4. The Corrected Loop (Using your UI discovery)
    for district in main_districts:
        print(f"-> Extracting: {district}")
        try:
            # Step A: Click Lagos (Because we are at the Root view)
            lagos_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(@class, 'b-region-item__name') and contains(text(), 'Lagos')]")))
            driver.execute_script("arguments[0].click();", lagos_btn)
            time.sleep(2)

            # Step B: Find and click the specific District (e.g., Ikorodu)
            target = wait.until(EC.presence_of_element_located((By.XPATH, f"//span[contains(@class, 'b-region-item__name') and text()='{district}']")))
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", target)
            time.sleep(1) 
            driver.execute_script("arguments[0].click();", target)
            time.sleep(2)

            # Step C: Extract Sub-locations
            sub_elements = driver.find_elements(By.CLASS_NAME, "b-region-item__name")
            for sub in sub_elements:
                sub_name = sub.text.strip()
                if sub_name and not sub_name.lower().startswith('all '):
                    master_records.append({
                        'Sub_Location': sub_name,
                        'Main_District': district,
                        'Full_Combined_Name': f"{sub_name} {district}"
                    })

            # Step D: Click Back (Which returns us all the way to Root)
            back_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'b-popup-header__back')] | //*[name()='svg' and contains(@class, 'back')] | //div[contains(text(), 'Back')]")))
            driver.execute_script("arguments[0].click();", back_btn)
            time.sleep(2)

        except Exception as e:
            print(f"   [!] Failed to extract {district}. Resetting state.")
            driver.refresh() # Hard reset if it gets lost
            time.sleep(4)
            location_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@filter-name='region'] | //div[contains(@class, 'filter') and contains(., 'Location')]")))
            location_btn.click()
            time.sleep(2)
            continue

    # 5. Save the data
    if master_records:
        df_master = pd.DataFrame(master_records).drop_duplicates()
        df_master.to_csv("Jiji_Official_Lagos_Dictionary.csv", index=False)
        print(f"\nSUCCESS: Extracted {len(df_master)} locations using the corrected UI flow!")
    else:
        print("\nExtraction finished, but no records were saved.")

except Exception as e:
    print(f"Critical Script Error: {e}")
finally:
    driver.quit()

Booting Selenium with the corrected navigation flow...
Opening location modal...
Mapping main districts...
Found 42 main districts. Starting extraction loop...
Critical Script Error: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x8bb593+105d3]
	chromedriver!GetHandleVerifier [0x8bb6c4+10704]
	chromedriver!(No symbol) [0x6c1ea0]
	chromedriver!(No symbol) [0x70b28a]
	chromedriver!(No symbol) [0x70b52b]
	chromedriver!(No symbol) [0x74d052]
	chromedriver!(No symbol) [0x72d7c4]
	chromedriver!(No symbol) [0x74a936]
	chromedriver!(No symbol) [0x72d516]
	chromedriver!(No symbol) [0x7008e9]
	chromedriver!(No symbol) [0x7016a4]
	chromedriver!GetHandleVerifier [0xb43014+298054]
	chromedriver!GetHandleVerifier [0xb3e603+293643]
	chromedriver!GetHandleVerifier [0xb5ea05+2b3a45]
	chromedriver!GetHandleVerifier [0x8d54e8+2a528]
	chromedriver!GetHandleVerifier [0x8dcd1d+31d5d]
	chromedriver!GetHandleVerifier [0x8c3e68+18ea8]
	chromedriver!GetHandleVerifier [0x8c4015+19055]
	chromedriver!GetH

In [11]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("Booting Selenium with Alphabetical strict-targeting...")

# Set to False if you want it to run invisibly again
# Leaving it commented out so you can watch it work based on your findings
options = webdriver.ChromeOptions()
# options.add_argument('--headless=new')
options.add_argument('--window-size=1920,1080')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 10)

master_records = []

try:
    # --- PRE-FLIGHT: GET THE MASTER LIST OF NAMES ONCE ---
    driver.get("https://jiji.ng/houses-apartments-for-rent")
    time.sleep(4)

    # Open Location -> Click Lagos
    location_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//div[@filter-name='region'] | //div[contains(@class, 'filter') and contains(., 'Location')]")))
    driver.execute_script("arguments[0].click();", location_btn)
    time.sleep(2)

    lagos_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(@class, 'b-region-item__name') and contains(text(), 'Lagos')]")))
    driver.execute_script("arguments[0].click();", lagos_btn)
    time.sleep(3)

    # Extract unique main district names
    district_elements = driver.find_elements(By.CLASS_NAME, "b-region-item__name")
    main_districts = [el.text.strip() for el in district_elements if el.text.strip() and not el.text.strip().lower().startswith('all ')]
    main_districts = sorted(list(set(main_districts)))
    
    print(f"Found {len(main_districts)} main districts. Starting hard-reset extraction loop...")

    # --- THE LOOP: Execute your exact findings ---
    for district in main_districts:
        print(f"-> Extracting: {district}")
        try:
            # 1. Hard reset to the base page
            driver.get("https://jiji.ng/houses-apartments-for-rent")
            time.sleep(3)

            # 2. Open Modal
            location_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//div[@filter-name='region'] | //div[contains(@class, 'filter') and contains(., 'Location')]")))
            driver.execute_script("arguments[0].click();", location_btn)
            time.sleep(2)

            # 3. Click Lagos
            lagos_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(@class, 'b-region-item__name') and contains(text(), 'Lagos')]")))
            driver.execute_script("arguments[0].click();", lagos_btn)
            time.sleep(2)

            # 4. Skip "Popular" and click the Alphabetical listing using [last()]
            target_xpath = f"(//span[contains(@class, 'b-region-item__name') and text()='{district}'])[last()]"
            target = wait.until(EC.presence_of_element_located((By.XPATH, target_xpath)))
            
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", target)
            time.sleep(1) 
            driver.execute_script("arguments[0].click();", target)
            time.sleep(2)

            # 5. Extract the sub-locations
            sub_elements = driver.find_elements(By.CLASS_NAME, "b-region-item__name")
            for sub in sub_elements:
                sub_name = sub.text.strip()
                if sub_name and not sub_name.lower().startswith('all '):
                    master_records.append({
                        'Sub_Location': sub_name,
                        'Main_District': district,
                        'Full_Combined_Name': f"{sub_name} {district}"
                    })
            print(f"   [+] Success!")

        except Exception as e:
            print(f"   [!] Failed to extract {district}. Skipping.")
            continue

    # --- SAVE THE DATA ---
    if master_records:
        df_master = pd.DataFrame(master_records).drop_duplicates()
        df_master.to_csv("Jiji_Official_Lagos_Dictionary.csv", index=False)
        print(f"\nSUCCESS: Extracted {len(df_master)} total sub-locations!")
    else:
        print("\nExtraction finished, but no records were saved.")

except Exception as e:
    print(f"Critical Script Error: {e}")
finally:
    driver.quit()

Booting Selenium with Alphabetical strict-targeting...
Found 42 main districts. Starting hard-reset extraction loop...
-> Extracting: Abule Egba
   [+] Success!
-> Extracting: Agbara-Igbesan
   [+] Success!
-> Extracting: Agboyi/Ketu
   [+] Success!
-> Extracting: Agege
   [+] Success!
-> Extracting: Ajah
   [+] Success!
-> Extracting: Alimosho
   [+] Success!
-> Extracting: Amuwo-Odofin
   [+] Success!
-> Extracting: Apapa
   [+] Success!
-> Extracting: Badagry
   [+] Success!
-> Extracting: Egbe/Idimu
   [+] Success!
-> Extracting: Ejigbo
   [+] Success!
-> Extracting: Eko Atlantic
   [+] Success!
-> Extracting: Epe
   [!] Failed to extract Epe. Skipping.
-> Extracting: Gbagada
   [!] Failed to extract Gbagada. Skipping.
-> Extracting: Ibeju
   [!] Failed to extract Ibeju. Skipping.
-> Extracting: Ifako-Ijaiye
   [!] Failed to extract Ifako-Ijaiye. Skipping.
-> Extracting: Ikeja
   [!] Failed to extract Ikeja. Skipping.
-> Extracting: Ikorodu
   [!] Failed to extract Ikorodu. Skippin

In [13]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("Booting automated Selenium extractor...")

options = webdriver.ChromeOptions()
# Running visibly so you can let it work in the background
options.add_argument('--window-size=1920,1080')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 10)

master_records = []

try:
    # --- PRE-FLIGHT: GET THE MASTER LIST OF NAMES ONCE ---
    driver.get("https://jiji.ng/houses-apartments-for-rent")
    time.sleep(4)

    # 1. Open Location Modal using your specific class name
    location_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".qa-choose-location")))
    driver.execute_script("arguments[0].click();", location_btn)
    time.sleep(2)

    # 2. Click Lagos
    lagos_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(@class, 'b-region-item__name') and contains(text(), 'Lagos')]")))
    driver.execute_script("arguments[0].click();", lagos_btn)
    time.sleep(3)

    # 3. Extract unique main district names
    district_elements = driver.find_elements(By.CLASS_NAME, "b-region-item__name")
    main_districts = [el.text.strip() for el in district_elements if el.text.strip() and not el.text.strip().lower().startswith('all ')]
    main_districts = sorted(list(set(main_districts)))
    
    print(f"Found {len(main_districts)} main districts. Starting hard-reset extraction loop...")

    # --- THE LOOP: Fully Automated ---
    for district in main_districts:
        print(f"-> Extracting: {district}")
        try:
            # A. Hard reset to the base page
            driver.get("https://jiji.ng/houses-apartments-for-rent")
            time.sleep(3)

            # B. Open Modal (using the new class)
            location_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".qa-choose-location")))
            driver.execute_script("arguments[0].click();", location_btn)
            time.sleep(2)

            # C. Click Lagos
            lagos_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(@class, 'b-region-item__name') and contains(text(), 'Lagos')]")))
            driver.execute_script("arguments[0].click();", lagos_btn)
            time.sleep(2)

            # D. Skip "Popular" and click the Alphabetical listing using [last()]
            target_xpath = f"(//span[contains(@class, 'b-region-item__name') and text()='{district}'])[last()]"
            target = wait.until(EC.presence_of_element_located((By.XPATH, target_xpath)))
            
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", target)
            time.sleep(1) 
            driver.execute_script("arguments[0].click();", target)
            time.sleep(2)

            # E. Extract the sub-locations
            sub_elements = driver.find_elements(By.CLASS_NAME, "b-region-item__name")
            for sub in sub_elements:
                sub_name = sub.text.strip()
                if sub_name and not sub_name.lower().startswith('all '):
                    master_records.append({
                        'Sub_Location': sub_name,
                        'Main_District': district,
                        'Full_Combined_Name': f"{sub_name} {district}"
                    })
            print(f"   [+] Success!")

        except Exception as e:
            print(f"   [!] Failed to extract {district}. Skipping.")
            continue

    # --- SAVE THE DATA ---
    if master_records:
        df_master = pd.DataFrame(master_records).drop_duplicates()
        df_master.to_csv("Jiji_Official_Lagos_Dictionary.csv", index=False)
        print(f"\nSUCCESS: Extracted {len(df_master)} total sub-locations!")
    else:
        print("\nExtraction finished, but no records were saved.")

except Exception as e:
    print(f"Critical Script Error: {e}")
finally:
    driver.quit()

Booting automated Selenium extractor...
Found 42 main districts. Starting hard-reset extraction loop...
-> Extracting: Abule Egba
   [+] Success!
-> Extracting: Agbara-Igbesan
   [+] Success!
-> Extracting: Agboyi/Ketu
   [+] Success!
-> Extracting: Agege
   [+] Success!
-> Extracting: Ajah
   [+] Success!
-> Extracting: Alimosho
   [+] Success!
-> Extracting: Amuwo-Odofin
   [+] Success!
-> Extracting: Apapa
   [+] Success!
-> Extracting: Badagry
   [+] Success!
-> Extracting: Egbe/Idimu
   [+] Success!
-> Extracting: Ejigbo
   [+] Success!
-> Extracting: Eko Atlantic
   [+] Success!
-> Extracting: Epe
   [+] Success!
-> Extracting: Gbagada
   [+] Success!
-> Extracting: Ibeju
   [+] Success!
-> Extracting: Ifako-Ijaiye
   [+] Success!
-> Extracting: Ikeja
   [+] Success!
-> Extracting: Ikorodu
   [+] Success!
-> Extracting: Ikotun/Igando
   [+] Success!
-> Extracting: Ikoyi
   [+] Success!
-> Extracting: Ilashe
   [+] Success!
-> Extracting: Ilupeju
   [+] Success!
-> Extracting: Ipa

In [1]:
import pandas as pd
from thefuzz import process

print("Loading datasets...")

# 1. Load the original scraped data and the new official dictionary
df_main = pd.read_csv("lagos_rent_cleaned_locations.csv")
target_col = 'Location' if 'Location' in df_main.columns else df_main.columns[1]

df_dict = pd.read_csv("Jiji_Official_Lagos_Dictionary.csv")
official_sub_locs = df_dict['Sub_Location'].astype(str).tolist()

# 2. Extract only the unique messy locations to save processing time
unique_messy_locs = df_main[target_col].dropna().unique()

mapping_dict = {}
print(f"Fuzzy matching {len(unique_messy_locs)} unique scraped locations against the Official Dictionary...")

# 3. Match the messy text to the clean Jiji dictionary
for messy in unique_messy_locs:
    messy_str = str(messy)
    
    # We use an 80% threshold to catch typos like "Ikoroddu" or extra words like "Plot 4 Agric"
    match = process.extractOne(messy_str, official_sub_locs, score_cutoff=80)
    
    if match:
        mapping_dict[messy_str] = match[0] # The exact official sub-location name
    else:
        mapping_dict[messy_str] = "Unmapped"

print("Applying matches to the full dataset...")

# 4. Map the newly matched clean names back onto all 12,000+ rows
df_main['Official_Sub_Location'] = df_main[target_col].astype(str).map(mapping_dict)

# 5. Merge the Main District info from the dictionary so your ML model knows the parent area
df_final = df_main.merge(
    df_dict[['Sub_Location', 'Main_District']], 
    how='left', 
    left_on='Official_Sub_Location', 
    right_on='Sub_Location'
)

# Clean up redundant columns from the merge
if 'Sub_Location' in df_final.columns:
    df_final.drop(columns=['Sub_Location'], inplace=True)

# 6. Save the pristine dataset
df_final.to_csv("lagos_rent_mapped_final.csv", index=False)

print("\n--- MAPPING COMPLETE ---")
print(f"Total Properties Processed: {len(df_final)}")
print(f"Successfully mapped rows: {len(df_final[df_final['Official_Sub_Location'] != 'Unmapped'])}")
print("Dataset securely saved to 'lagos_rent_mapped_final.csv'")

Loading datasets...
Fuzzy matching 3147 unique scraped locations against the Official Dictionary...
Applying matches to the full dataset...

--- MAPPING COMPLETE ---
Total Properties Processed: 27090
Successfully mapped rows: 27055
Dataset securely saved to 'lagos_rent_mapped_final.csv'


In [2]:
import pandas as pd
from thefuzz import process

print("Loading datasets and enforcing unique merge keys...")

# 1. Load the original scraped data
df_main = pd.read_csv("lagos_rent_cleaned_locations.csv")
target_col = 'Location' if 'Location' in df_main.columns else df_main.columns[1]

# 2. Load the dictionary AND strictly deduplicate the Sub-Locations
df_dict = pd.read_csv("Jiji_Official_Lagos_Dictionary.csv")

# THE FIX: This stops Pandas from multiplying rows if names like "GRA" appear in multiple districts
df_dict_unique = df_dict.drop_duplicates(subset=['Sub_Location'], keep='first')
official_sub_locs = df_dict_unique['Sub_Location'].astype(str).tolist()

# 3. Extract unique messy locations
unique_messy_locs = df_main[target_col].dropna().unique()
mapping_dict = {}

print(f"Fuzzy matching {len(unique_messy_locs)} unique scraped locations...")

# 4. Match the messy text to the clean, unique list
for messy in unique_messy_locs:
    messy_str = str(messy)
    match = process.extractOne(messy_str, official_sub_locs, score_cutoff=80)
    
    if match:
        mapping_dict[messy_str] = match[0]
    else:
        mapping_dict[messy_str] = "Unmapped"

print("Applying matches and merging safely...")

# 5. Map the clean names back to the main dataset
df_main['Official_Sub_Location'] = df_main[target_col].astype(str).map(mapping_dict)

# 6. Merge safely using the deduplicated dictionary
df_final = df_main.merge(
    df_dict_unique[['Sub_Location', 'Main_District']], 
    how='left', 
    left_on='Official_Sub_Location', 
    right_on='Sub_Location'
)

# Clean up the extra column
if 'Sub_Location' in df_final.columns:
    df_final.drop(columns=['Sub_Location'], inplace=True)

# 7. Save the properly sized dataset
df_final.to_csv("lagos_rent_mapped_final.csv", index=False)

print("\n--- MAPPING COMPLETE ---")
print(f"Original Properties Before Script: {len(df_main)}")
print(f"Final Properties After Script: {len(df_final)}")

if len(df_main) == len(df_final):
    print("SUCCESS: Row count maintained! Cartesian explosion prevented.")
else:
    print("WARNING: Row count still mismatched. Check for deeper dataset duplications.")

Loading datasets and enforcing unique merge keys...
Fuzzy matching 3147 unique scraped locations...
Applying matches and merging safely...

--- MAPPING COMPLETE ---
Original Properties Before Script: 12075
Final Properties After Script: 12075
SUCCESS: Row count maintained! Cartesian explosion prevented.


In [3]:
import pandas as pd

print("Generating readable text trees...")

# Load the datasets
df_dict = pd.read_csv("Jiji_Official_Lagos_Dictionary.csv")
df_main = pd.read_csv("lagos_rent_mapped_final.csv")
target_col = 'Location' if 'Location' in df_main.columns else df_main.columns[1]

# ---------------------------------------------------------
# FILE 1: The Clean Dictionary Tree
# ---------------------------------------------------------
with open("Lagos_Dictionary_Tree.txt", "w", encoding="utf-8") as f:
    f.write("=== OFFICIAL LAGOS LOCATION DICTIONARY ===\n\n")
    
    # Group by Main District
    grouped_dict = df_dict.groupby('Main_District')
    for district, group in sorted(grouped_dict):
        f.write(f"📍 {district}\n")
        
        # Sort sub-locations alphabetically
        sub_locs = sorted(group['Sub_Location'].astype(str).unique())
        for sub in sub_locs:
            f.write(f"    ├── {sub}\n")
        f.write("\n")

print("-> Created 'Lagos_Dictionary_Tree.txt'")

# ---------------------------------------------------------
# FILE 2: The Mapping Traceability Tree
# ---------------------------------------------------------
with open("Lagos_Mapped_Strings_Tree.txt", "w", encoding="utf-8") as f:
    f.write("=== HOW MESSY STRINGS MAPPED TO OFFICIAL LOCATIONS ===\n\n")
    
    # We only want rows that successfully mapped
    df_mapped = df_main[df_main['Official_Sub_Location'] != 'Unmapped']
    
    # Group by Main District and Official Sub-Location
    grouped_main = df_mapped.groupby(['Main_District', 'Official_Sub_Location'])
    
    current_district = ""
    for (district, sub_loc), group in sorted(grouped_main):
        if district != current_district:
            f.write(f"\n📍 {district}\n")
            f.write(f"{'='*len(district) + '===='}\n")
            current_district = district
        
        f.write(f"  📂 {sub_loc}\n")
        
        # Get unique messy strings that mapped to this sub-location
        messy_strings = sorted(group[target_col].astype(str).unique())
        for messy in messy_strings:
            f.write(f"      ├── {messy}\n")
        f.write("\n")

print("-> Created 'Lagos_Mapped_Strings_Tree.txt'")
print("\nSUCCESS: Both files are ready to view in your project folder!")

Generating readable text trees...
-> Created 'Lagos_Dictionary_Tree.txt'
-> Created 'Lagos_Mapped_Strings_Tree.txt'

SUCCESS: Both files are ready to view in your project folder!


In [4]:
import pandas as pd
import re
from thefuzz import process

print("Loading datasets with structural constraints...")
df_main = pd.read_csv("lagos_rent_cleaned_locations.csv")
target_col = 'Location' if 'Location' in df_main.columns else df_main.columns[1]

df_dict = pd.read_csv("Jiji_Official_Lagos_Dictionary.csv")
df_dict_unique = df_dict.drop_duplicates(subset=['Sub_Location'], keep='first')

# THE FIX: Sort official locations longest-to-shortest so "Lekki Phase 1" is checked BEFORE "Lekki"
official_sub_locs = df_dict_unique['Sub_Location'].astype(str).tolist()
official_sub_locs.sort(key=len, reverse=True)

unique_messy_locs = df_main[target_col].dropna().unique()
mapping_dict = {}

print("Executing Length-Sorted Proximity Mapper...")

def find_smart_match(address, sorted_official_list):
    address_clean = str(address).strip().title()
    
    # --- STEP 1: LONGEST-STRING EXACT MATCHING (Solves Phase 1 / Phase 2) ---
    exact_matches = []
    for loc in sorted_official_list:
        # Check for standalone word block matching using regex
        if re.search(r'\b' + re.escape(loc) + r'\b', address_clean, re.IGNORECASE):
            exact_matches.append(loc)
            
    # If we found exact matches, we need to make sure shorter words don't hijack longer phases
    if exact_matches:
        # Filter exact matches to keep only the longest ones that aren't sub-strings of each other
        # e.g., if we matched "Lekki Phase 1" and "Lekki", drop "Lekki"
        longest_matches = []
        for match in exact_matches:
            if not any(match != other and match.lower() in other.lower() for other in exact_matches):
                longest_matches.append(match)
                
        if len(longest_matches) == 1:
            return longest_matches[0]
        
        # --- STEP 2: PROXIMITY ANCHORING (If multiple distinct locations match, e.g., LSDPC and Lekki) ---
        if len(longest_matches) > 1:
            best_match = None
            min_distance_from_end = float('inf')
            
            for match in longest_matches:
                match_index = address_clean.lower().rfind(match.lower())
                distance = len(address_clean) - match_index
                
                if distance < min_distance_from_end:
                    min_distance_from_end = distance
                    best_match = match
            return best_match

    # --- STEP 3: TAIL-END SEGMENT FUZZY MATCHING (Fallback) ---
    tail_cleaned = re.sub(r'\b(lagos state|lagos|nigeria)\b', '', address_clean, flags=re.IGNORECASE).strip()
    segments = [seg.strip() for seg in tail_cleaned.split(',') if seg.strip()]
    if segments:
        target_zone = " ".join(segments[-2:])
        match = process.extractOne(target_zone, sorted_official_list, score_cutoff=80)
        if match:
            return match[0]

    # Global global fallback
    match = process.extractOne(address_clean, sorted_official_list, score_cutoff=80)
    return match[0] if match else "Unmapped"


# Run the mapping across all unique strings
for messy in unique_messy_locs:
    mapping_dict[str(messy)] = find_smart_match(messy, official_sub_locs)

print("Applying changes and writing clean CSV output...")
df_main['Official_Sub_Location'] = df_main[target_col].astype(str).map(mapping_dict)

df_final = df_main.merge(
    df_dict_unique[['Sub_Location', 'Main_District']], 
    how='left', 
    left_on='Official_Sub_Location', 
    right_on='Sub_Location'
)

if 'Sub_Location' in df_final.columns:
    df_final.drop(columns=['Sub_Location'], inplace=True)

df_final.to_csv("lagos_rent_mapped_final.csv", index=False)
print("SUCCESS: Master geography is completely clean and ready for your ML models!")

Loading datasets with structural constraints...
Executing Length-Sorted Proximity Mapper...
Applying changes and writing clean CSV output...
SUCCESS: Master geography is completely clean and ready for your ML models!


In [5]:
import pandas as pd

print("Generating readable text trees from the updated mapping...")

# Load the datasets
df_dict = pd.read_csv("Jiji_Official_Lagos_Dictionary.csv")
df_main = pd.read_csv("lagos_rent_mapped_final.csv")
target_col = 'Location' if 'Location' in df_main.columns else df_main.columns[1]

# ---------------------------------------------------------
# FILE 1: The Clean Dictionary Tree
# ---------------------------------------------------------
with open("Lagos_Dictionary_Tree.txt", "w", encoding="utf-8") as f:
    f.write("=== OFFICIAL LAGOS LOCATION DICTIONARY ===\n\n")
    
    # Group by Main District
    grouped_dict = df_dict.groupby('Main_District')
    for district, group in sorted(grouped_dict):
        f.write(f"📍 {district}\n")
        
        # Sort sub-locations alphabetically
        sub_locs = sorted(group['Sub_Location'].astype(str).unique())
        for sub in sub_locs:
            f.write(f"    ├── {sub}\n")
        f.write("\n")

print("-> Created 'Lagos_Dictionary_Tree.txt'")

# ---------------------------------------------------------
# FILE 2: The Mapping Traceability Tree
# ---------------------------------------------------------
with open("Lagos_Mapped_Strings_Tree.txt", "w", encoding="utf-8") as f:
    f.write("=== HOW MESSY STRINGS MAPPED TO OFFICIAL LOCATIONS ===\n\n")
    
    # We only want rows that successfully mapped
    df_mapped = df_main[df_main['Official_Sub_Location'] != 'Unmapped']
    
    # Group by Main District and Official Sub-Location
    grouped_main = df_mapped.groupby(['Main_District', 'Official_Sub_Location'])
    
    current_district = ""
    for (district, sub_loc), group in sorted(grouped_main):
        if district != current_district:
            f.write(f"\n📍 {district}\n")
            f.write(f"{'='*len(district) + '===='}\n")
            current_district = district
        
        f.write(f"  📂 {sub_loc}\n")
        
        # Get unique messy strings that mapped to this sub-location
        messy_strings = sorted(group[target_col].astype(str).unique())
        for messy in messy_strings:
            f.write(f"      ├── {messy}\n")
        f.write("\n")

print("-> Created 'Lagos_Mapped_Strings_Tree.txt'")
print("\nSUCCESS: Trees generated! Open 'Lagos_Mapped_Strings_Tree.txt' to verify the Lekki splits.")

Generating readable text trees from the updated mapping...
-> Created 'Lagos_Dictionary_Tree.txt'
-> Created 'Lagos_Mapped_Strings_Tree.txt'

SUCCESS: Trees generated! Open 'Lagos_Mapped_Strings_Tree.txt' to verify the Lekki splits.
